In [ ]:
import os
import faiss
from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    StorageContext,
    Settings
)
from llama_index.vector_stores.faiss import FaissVectorStore

# Asegurar que las rutas apuntan a la raíz del proyecto
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(BASE_DIR, "data", "raw_pdfs")
INDEX_DIR = os.path.join(BASE_DIR, "data", "vector_store")

os.makedirs(INDEX_DIR, exist_ok=True)

In [ ]:
# Carga todos los PDFs de la carpeta raw_pdfs
print(f"Cargando documentos desde: {DATA_DIR}")
documents = SimpleDirectoryReader(DATA_DIR).load_data()
print(f"Se han cargado {len(documents)} fragmentos/páginas.")

# Aquí puedes añadir lógica de metadatos si trabajas con boletines oficiales.
# Si el modelo busca un código de actividad o zona que no existe en el documento, 
# la configuración del RAG en el agente deberá forzar un retorno nulo o "no encontrado" 
# en lugar de aproximar resultados.

In [ ]:
# Dimensión para el modelo de embeddings por defecto de LlamaIndex (OpenAI/Gemini)
# Ajusta el número 1536 si usas un modelo diferente de embeddings.
d = 1536 
faiss_index = faiss.IndexFlatL2(d)

# Configurar el vector store de LlamaIndex con el índice FAISS subyacente
vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Crear el índice a partir de los documentos
index = VectorStoreIndex.from_documents(
    documents, 
    storage_context=storage_context
)
print("Índice vectorial creado con éxito.")

In [ ]:
# Guardar el índice en disco para que el Agente Normativo pueda consumirlo sin re-indexar
index.storage_context.persist(persist_dir=INDEX_DIR)
print(f"Índice FAISS guardado correctamente en: {INDEX_DIR}")